# Synthetic Control File for Employee Records
- Goal: Generate synthetic employee records for testing and development purposes.

## Imports

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from datetime import date, datetime
from typing import Dict, List, Optional, Tuple, Literal

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Configs
- This configuration will drive downstream generators (employees -> payroll -> GL).
- Keep config objects typed and deterministic (seeded) for reproducibility.

In [ ]:
LocationStrategy = Literal["NY", "SF", "REMOTE"]
JobFamily = Literal["ENGINEERING", "SALES", "PRODUCT", "G&A"]
EmailUniquenessStrategy = Literal["append_id", "dedupe_counter"]

In [ ]:
@dataclass(frozen=True)
class CompanyConfig:
    company_name: str = "Gamma Software Solutions, Inc."
    company_domain: str = "gammasoftware.com"
    entity_id: str = "ENT100"
    currency: str = "USD"


@dataclass(frozen=True)
class TimeConfig:
    start_date: date = date(2020, 1, 1)
    as_of_date: date = date(2026, 1, 1)


@dataclass(frozen=True)
class PopulationConfig:
    start_headcount: int = 3
    target_headcount: int = 105


@dataclass(frozen=True)
class IdConfig:
    employee_id_prefix: str = "EMP"
    employee_id_width: int = 6


@dataclass(frozen=True)
class PolicyConfig:
    include_terminations: bool = False
    termination_rate_annual: float = 0.0
    email_uniqueness_strategy: EmailUniquenessStrategy = "append_id"


@dataclass(frozen=True)
class DistributionConfig:
    # high-level organizational job family mix
    job_family_weights: Dict[JobFamily, float] = field(
        default_factory = lambda: {
            "ENGINEERING": 0.5,
            "SALES": 0.15,
            "PRODUCT": 0.15,
            "G&A": 0.2
        }
    )

    # location mix
    location_weights: Dict[LocationStrategy, float] = field(
        default_factory = lambda: {
            "NY": 0.4,
            "SF": 0.3,
            "REMOTE": 0.3
        }
    )

    # job family levels (L1 to L6)
    level_weights_by_family: Dict[JobFamily, Dict[str, float]] = field(
        default_factory = lambda: {
            "ENGINEERING": {"L1": 0.10, "L2": 0.20, "L3": 0.25, "L4": 0.25, "L5": 0.15, "L6": 0.05},
            "SALES": {"L1": 0.15, "L2": 0.25, "L3": 0.25, "L4": 0.20, "L5": 0.10, "L6": 0.05},
            "PRODUCT": {"L1": 0.10, "L2": 0.25, "L3": 0.25, "L4": 0.20, "L5": 0.15, "L6": 0.05},
            "G&A": {"L1": 0.20, "L2": 0.30, "L3": 0.25, "L4": 0.15, "L5": 0.08, "L6": 0.02},
        }
    )


@dataclass(frozen=True)
class RunConfig:
    seed: int = 42
    source: str = "SYNTHETIC_V1"
    created_at: datetime = field(default_factory=lambda: datetime.now())


@dataclass(frozen=True)
class GlobalConfig:
    company: CompanyConfig = field(default_factory=CompanyConfig)
    time: TimeConfig = field(default_factory=TimeConfig)
    population: PopulationConfig = field(default_factory=PopulationConfig)
    ids: IdConfig = field(default_factory=IdConfig)
    policy: PolicyConfig = field(default_factory=PolicyConfig)
    dist: DistributionConfig = field(default_factory=DistributionConfig)
    run: RunConfig = field(default_factory=RunConfig)


CONFIG = GlobalConfig()
CONFIG